In [ ]:
#concept of an Offline Markov Decision Process (MDP) / Offline Reinforcement Learning context,
# specifically applied to the problem of data imputation.


import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

# 1. Load the original file 
df_true = pd.read_csv("heart.csv")

# Ensure logical zero anomalies are flagged as NaN for correction
df_true["Cholesterol"] = df_true["Cholesterol"].replace(0, np.nan)
df_true["RestingBP"] = df_true["RestingBP"].replace(0, np.nan)

# 2. Replicate artificial corruption setup (15% Cholesterol anomalies)
np.random.seed(42)
df_corrupted = df_true.copy()
mask = np.random.rand(len(df_true)) < 0.15
df_corrupted.loc[mask, "Cholesterol"] = np.nan

# Calculate historical population medians (X-tilde) from available non-missing entries
median_chol = df_true["Cholesterol"].median()
median_bp = df_true["RestingBP"].median()

# 3. Handle Categorical Columns using Mode Imputation to stabilize state feature vectors
categorical_cols = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]
cat_imputer = SimpleImputer(strategy='most_frequent')
df_corrupted[categorical_cols] = cat_imputer.fit_transform(df_corrupted[categorical_cols])

# Generate Numeric Vector Space for the State Layout
df_numeric_corrupted = pd.get_dummies(df_corrupted, columns=categorical_cols, drop_first=True)

# =====================================================================
# REINFORCEMENT LEARNING ENVIRONMENT STEP EXECUTION LOOP (L1 DISTANCE)
# =====================================================================
df_cleaned_rl = df_corrupted.copy()
total_accumulated_reward = 0
imputation_records = []

# Define localized scaling penalties as per your formulation
lambda_chol = 0.2
lambda_bp = 0.3
R_max = 100

print("🤖 RL Agent processing transitions and computing L1 error rewards...")

for idx in range(len(df_numeric_corrupted)):
    # Identify missing values / anomalies in the state space
    is_chol_missing = np.isnan(df_corrupted.loc[idx, "Cholesterol"])
    is_bp_missing = np.isnan(df_corrupted.loc[idx, "RestingBP"])
    
    if is_chol_missing or is_bp_missing:
        # 1. Observe State Vector (known features for this trajectory step)
        state = df_numeric_corrupted.loc[idx, :].values
        
        # 2. Simulate Agent Policy Action Selection (Guesses)
        # If value is missing, the agent policy makes a continuous guess, else retains true value
        guessed_chol = int(round(np.random.normal(loc=240, scale=30))) if is_chol_missing else df_true.loc[idx, "Cholesterol"]
        guessed_bp = int(round(np.random.normal(loc=130, scale=15))) if is_bp_missing else df_true.loc[idx, "RestingBP"]
        
        # 3. Compute Internal Reward Function (Ri) via the Linear Absolute Error Penalty Model
        # L1 Manhattan distances from the historical population median
        dist_chol = abs(guessed_chol - median_chol) if is_chol_missing else 0
        dist_bp = abs(guessed_bp - median_bp) if is_bp_missing else 0
        
        # Ri = Rmax - (lambda_chol * |X_chol,i - X_tilde_chol| + lambda_bp * |X_bp,i - X_tilde_bp|)
        reward_i = R_max - ((lambda_chol * dist_chol) + (lambda_bp * dist_bp))
        total_accumulated_reward += reward_i
        
        # Commit actions back to the dataset
        df_cleaned_rl.loc[idx, "Cholesterol"] = guessed_chol
        df_cleaned_rl.loc[idx, "RestingBP"] = guessed_bp
        
        imputation_records.append({
            "Row": idx,
            "Guessed_Chol": guessed_chol if is_chol_missing else "Retained",
            "Guessed_BP": guessed_bp if is_bp_missing else "Retained",
            "L1_Chol_Dist": dist_chol,
            "L1_BP_Dist": dist_bp,
            "Internal_Reward_Ri": round(reward_i, 2)
        })

# Finalize formatting and ensure integer types
df_cleaned_rl["Cholesterol"] = df_cleaned_rl["Cholesterol"].fillna(median_chol).round().astype(int)
df_cleaned_rl["RestingBP"] = df_cleaned_rl["RestingBP"].fillna(median_bp).round().astype(int)

# 4. Export finalized datasets
df_cleaned_rl.to_csv("heart_rl_cleaned.csv", index=False)
df_cleaned_rl.to_csv("fill_miss.txt", sep="\t", index=False)

# Display tabular simulation log
df_rewards = pd.DataFrame(imputation_records)
print("\n📊 First 15 Reinforcement Learning Transition Evaluations (Ri Metrics):")
print(df_rewards.head(15).to_string(index=False))
print(f"\n🏆 Total Accumulated Reward across trajectory space: {round(total_accumulated_reward, 2)}")



🤖 RL Agent processing transitions and computing L1 error rewards...

📊 First 15 Reinforcement Learning Transition Evaluations (Ri Metrics):
 Row Guessed_Chol Guessed_BP  L1_Chol_Dist  L1_BP_Dist  Internal_Reward_Ri
   6          224   Retained          13.0         0.0                97.4
  10          252   Retained          15.0         0.0                97.0
  21          239   Retained           2.0         0.0                99.6
  29          273   Retained          36.0         0.0                92.8
  32          243   Retained           6.0         0.0                98.8
  37          245   Retained           8.0         0.0                98.4
  40          229   Retained           8.0         0.0                98.4
  42          238   Retained           1.0         0.0                99.8
  56          249   Retained          12.0         0.0                97.6
  58          189   Retained          48.0         0.0                90.4
  59     Retained        110       